# EV Range Prediction — TechTrack 3.0
Specification-based regression solution.

In [ ]:

# EV Range Prediction — TechTrack 3.0
# Run from the project root.

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

DATA_PATH = "data/EvRangePredictionDataset.xlsx"
df = pd.read_excel(DATA_PATH)
df.shape


In [ ]:

# 1. Data quality profile
display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("Duplicate rows:", df.duplicated().sum())

# Known inconsistent cargo entries
print("Cargo entries requiring parsing:",
      df.loc[pd.to_numeric(df["cargo_volume_l"], errors="coerce").isna(),
             "cargo_volume_l"].tolist())

# 2. Clean cargo volume
def clean_cargo(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"\d+(?:\.\d+)?", str(value))
    return float(match.group()) if match else np.nan

df["cargo_volume_l"] = df["cargo_volume_l"].apply(clean_cargo)

# 3. EDA
fig = plt.figure(figsize=(8, 5))
plt.hist(df["range_km"], bins=20)
plt.xlabel("Range (km)")
plt.ylabel("Count")
plt.title("Distribution of EV range")
plt.show()

numeric = df.select_dtypes(include=np.number)
corr = numeric.corr(numeric_only=True)["range_km"].sort_values(ascending=False)
display(corr.to_frame("correlation_with_range"))

fig = plt.figure(figsize=(7, 5))
plt.scatter(df["battery_capacity_kWh"], df["range_km"], alpha=0.7)
plt.xlabel("Battery capacity (kWh)")
plt.ylabel("Range (km)")
plt.title("Battery capacity vs range")
plt.show()

display(df.groupby("drivetrain")["range_km"].agg(["count","mean","median"]).sort_values("mean", ascending=False))

# 4. Feature engineering and leakage-safe feature set
def prepare_features(frame):
    data = frame.copy()
    drop_cols = ["range_km", "efficiency_wh_per_km", "source_url", "model", "battery_type"]
    data = data.drop(columns=drop_cols, errors="ignore")
    data["footprint_m2"] = data["length_mm"] * data["width_mm"] / 1_000_000
    data["volume_proxy_m3"] = data["length_mm"] * data["width_mm"] * data["height_mm"] / 1_000_000_000
    data["battery_per_torque"] = data["battery_capacity_kWh"] / (data["torque_nm"].abs() + 1)
    data["battery_per_footprint"] = data["battery_capacity_kWh"] / (data["footprint_m2"] + 1e-6)
    data["performance_index"] = data["top_speed_kmh"] / (data["acceleration_0_100_s"] + 0.1)
    for col in data.select_dtypes(include="object").columns:
        data[col] = data[col].fillna("Missing").astype(str)
    return data

X = prepare_features(df)
y = df["range_km"]

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

cat_cols = X.select_dtypes(include="object").columns.tolist()
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

# 6. Compare multiple regression models
# The final submission uses CatBoost because it handles mixed numerical/categorical data
# well on a small dataset and avoids a large manual one-hot matrix.
model = CatBoostRegressor(
    iterations=600, depth=6, learning_rate=0.04,
    loss_function="RMSE", l2_leaf_reg=5,
    random_seed=42, verbose=False
)
model.fit(X_train, y_train, cat_features=[X_train.columns.get_loc(c) for c in cat_cols])
pred = model.predict(X_test)

metrics = pd.Series({
    "MAE": mean_absolute_error(y_test, pred),
    "RMSE": mean_squared_error(y_test, pred) ** 0.5,
    "R2": r2_score(y_test, pred)
})
display(metrics)

# 7. Five-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for fold, (tr, va) in enumerate(kf.split(X), 1):
    m = CatBoostRegressor(
        iterations=600, depth=6, learning_rate=0.04,
        loss_function="RMSE", l2_leaf_reg=5,
        random_seed=42, verbose=False
    )
    m.fit(X.iloc[tr], y.iloc[tr],
          cat_features=[X.iloc[tr].columns.get_loc(c) for c in cat_cols])
    p = m.predict(X.iloc[va])
    rows.append({
        "fold": fold,
        "MAE": mean_absolute_error(y.iloc[va], p),
        "RMSE": mean_squared_error(y.iloc[va], p) ** 0.5,
        "R2": r2_score(y.iloc[va], p)
    })
cv_results = pd.DataFrame(rows)
display(cv_results)
display(cv_results.mean(numeric_only=True))

# 8. Feature importance
importance = pd.Series(model.get_feature_importance(), index=X.columns).sort_values(ascending=False)
display(importance.head(15))

# 9. Final model artifact
import joblib
final_model = CatBoostRegressor(
    iterations=600, depth=6, learning_rate=0.04,
    loss_function="RMSE", l2_leaf_reg=5,
    random_seed=42, verbose=False
)
final_model.fit(X, y, cat_features=cat_idx)
joblib.dump(final_model, "models/ev_range_catboost.joblib")
print("Saved models/ev_range_catboost.joblib")
